# Lead assignment (notebook version)

Notebook equivalent of `assign_leads.py`: reads the new leads and the active sellers,
assigns each lead to a seller based on **language** and **content of interest**, and
materializes the result in `lead_assignments` (ClickHouse, `DB_NAME` database).

```
leads_clean  +  sales_people  ->  lead_assignments
```

All the logic lives in the `utils/` package; this notebook only orchestrates the steps.

## 1. Imports

`assign_leads.py` uses package-relative imports (`from .utils.db import ...`). Those only work
inside a module that belongs to a package, and a notebook is not one, so `from .utils ...` raises
`ImportError: attempted relative import with no known parent package`.

So we put this folder on `sys.path` and import `utils` from it. The working directory is not always
this folder: Jupyter uses it, but a scheduler running papermill (like Intelek) may start from the repo
root. The next cell looks for `utils/` from the working directory, `python_ipynb/` and
`compute/python_ipynb/`. Inside `utils/`, the
modules still use relative imports between themselves (`from .candidates import ...`).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# A notebook does not know where its own file is, and the working directory depends on who runs it:
# Jupyter uses the notebook's folder, a scheduler (papermill) may use the repo root.
# Find the folder that contains utils/ from any of them.
CANDIDATES = [Path.cwd(), Path.cwd() / "python_ipynb", Path.cwd() / "compute" / "python_ipynb"]
NOTEBOOK_DIR = next((d for d in CANDIDATES if (d / "utils" / "db.py").is_file()), None)
if NOTEBOOK_DIR is None:
    raise ModuleNotFoundError(f"utils/ not found from {Path.cwd()}: run from the repo root, "
                              "compute/ or compute/python_ipynb/")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.db import get_client, read_table, table_exists, write_table
from utils.matching import assign_leads, current_load, normalize_sellers, summarize

## 2. Parameters

The next cell is tagged `parameters`: when Intelek runs the notebook with papermill, it
injects a cell right after it with the actual values (including the `creds` secret).

In [ ]:
# Intelek injects `creds` (the secret object, a dict) in the first cell of the notebook.
# Keep it if it is already defined; otherwise (local runs) None -> read compute/.env
try:
    creds
except NameError:
    creds = None
LEADS_TABLE = "leads_clean"
SELLERS_TABLE = "sales_people"
OUTPUT_TABLE = "lead_assignments"

FULL_REFRESH = False   # True: ignore previous assignments and rebuild the table
DRY_RUN = False        # True: compute but do not write

In [ ]:
client = get_client(creds)

## 3. Read inputs

In [ ]:
leads = read_table(client, LEADS_TABLE)
sellers = normalize_sellers(read_table(client, SELLERS_TABLE))

print(f"{len(leads)} leads | {len(sellers)} active sellers")
sellers

## 4. Keep only the new leads

A lead is *new* if it is not in the output table yet. With `FULL_REFRESH = True` every lead is reassigned.

In [ ]:
existing = None
if not FULL_REFRESH and table_exists(client, OUTPUT_TABLE):
    existing = read_table(client, OUTPUT_TABLE)
    leads = leads[~leads["lead_id"].isin(existing["lead_id"])]

load = current_load(existing)   # {seller_id: leads already assigned}
print(f"{len(leads)} new leads | {0 if existing is None else len(existing)} previous assignments")
load

## 5. Assign

Rules, lead by lead in order of arrival:
1. Prefer sellers who speak the lead's language **and** are specialized in its content (`full_match`).
2. Otherwise, any seller speaking the language (`language_only`).
3. Within each tier, the seller with the lowest load / capacity ratio wins. If nobody is available the lead stays `unassigned`.

In [ ]:
assignments = assign_leads(leads, sellers, load=load)
summarize(assignments)

In [ ]:
# Leads that did not get a perfect match, and why
assignments[assignments["match_type"] != "full_match"]

## 6. Materialize

In [ ]:
if assignments.empty:
    print("Nothing to assign.")
elif DRY_RUN:
    print("Dry run: nothing written.")
else:
    write_table(assignments, client, OUTPUT_TABLE, if_exists="replace" if FULL_REFRESH else "append")
    print(f"{len(assignments)} rows written to {OUTPUT_TABLE}")

## 7. Check the output table

In [ ]:
client.query_df(f"""
    SELECT coalesce(seller_id, '-') AS seller_id, match_type, count(*) AS n_leads
    FROM {OUTPUT_TABLE}
    GROUP BY 1, 2
    ORDER BY 1, 2
""")